In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils, paths
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"
# Which result pathways this (location, vehicle) combo has. Snakemake passes
# these from config_utils.get_combo_pathways, the same function that decides
# which inputs the rule declares. A pathway the combo lacks contributes zeros,
# built from the template combo (india/rice). Never branch on whether a file
# exists: a job can start before the file is built, and a file missing by
# mistake would silently become zeros.
has_simulations = True
has_anemia_model = True
has_ntd_model = True
data_source = paths.data_source(default="local")  # or a model number on the team drive

In [3]:
# Parameters
location = "india"
vehicle = "rice"
has_simulations = True
has_anemia_model = True
has_ntd_model = True


In [4]:
rescaled_pregnancy_root = paths.source_root(paths.RESCALED_PREGNANCY_RESULTS_ROOT, data_source)
rescaled_child_root = paths.source_root(paths.RESCALED_CHILD_RESULTS_ROOT, data_source)
anemia_root = paths.source_root(paths.NON_PREGNANT_ANEMIA_RESULTS_ROOT, data_source)
print(f"{data_source}:\n  {rescaled_pregnancy_root}\n  {rescaled_child_root}\n  {anemia_root}")

model2.6.3:
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.3/rescaled_pregnancy
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.3/rescaled_child
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.3/non_pregnant_anemia


In [5]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [6]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/person_time_anemia.parquet"
if has_simulations:
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,141,13.687242
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,141,4.700669
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,141,8.710063
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,141,10.645632
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,141,8.387468
...,...,...,...,...,...,...,...,...,...,...,...
1079995,person_time,impairment,anemia,severe,95_plus,severe,1,zero,0,129,0.000000
1079996,person_time,impairment,anemia,severe,95_plus,severe,2,zero,0,129,0.000000
1079997,person_time,impairment,anemia,severe,95_plus,severe,3,zero,0,129,0.000000
1079998,person_time,impairment,anemia,severe,95_plus,severe,4,zero,0,129,0.000000


In [7]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [8]:
pregnancy_person_time_anemia.sub_entity.value_counts()

sub_entity
not_anemic    270000
mild          270000
moderate      270000
severe        270000
Name: count, dtype: int64

In [9]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  4.931144e+06
              2                  4.078597e+06
              3                  3.655063e+06
              4                  3.475287e+06
              5                  3.393213e+06
intervention  1                  4.931159e+06
              2                  4.078601e+06
              3                  3.655071e+06
              4                  3.475294e+06
              5                  3.393217e+06
zero          1                  4.931123e+06
              2                  4.078587e+06
              3                  3.655052e+06
              4                  3.475273e+06
              5                  3.393210e+06
Name: value, dtype: float64

In [10]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.676816e+06
              2                  2.115416e+06
              3                  1.796642e+06
              4                  1.579707e+06
              5                  1.333709e+06
intervention  1                  2.602695e+06
              2                  2.054884e+06
              3                  1.737096e+06
              4                  1.525993e+06
              5                  1.285803e+06
zero          1                  2.824931e+06
              2                  2.220461e+06
              3                  1.891263e+06
              4                  1.664768e+06
              5                  1.377133e+06
Name: value, dtype: float64

In [11]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.542839
              2                  0.518663
              3                  0.491549
              4                  0.454554
              5                  0.393052
intervention  1                  0.527806
              2                  0.503821
              3                  0.475257
              4                  0.439097
              5                  0.378933
zero          1                  0.572878
              2                  0.544419
              3                  0.517438
              4                  0.479032
              5                  0.405850
Name: value, dtype: float64

In [12]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [13]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,48678.711021
1,Female,0.0,0.019178,not_pregnant,2,42723.422230
2,Female,0.0,0.019178,not_pregnant,3,37932.716572
3,Female,0.0,0.019178,not_pregnant,4,35728.167677
4,Female,0.0,0.019178,not_pregnant,5,28631.476092
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,18328.880607
281,Male,95.0,125.000000,not_pregnant,2,19140.234171
282,Male,95.0,125.000000,not_pregnant,3,19770.250303
283,Male,95.0,125.000000,not_pregnant,4,20851.307187


In [14]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    4.478246e+06
2    3.707329e+06
3    3.317141e+06
4    3.157155e+06
5    3.080667e+06
Name: value, dtype: float64

In [15]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.430966e+06
              2                  1.922854e+06
              3                  1.630537e+06
              4                  1.435099e+06
              5                  1.210862e+06
intervention  1                  2.363645e+06
              2                  1.867830e+06
              3                  1.576493e+06
              4                  1.386299e+06
              5                  1.167367e+06
zero          1                  2.565488e+06
              2                  2.018341e+06
              3                  1.716415e+06
              4                  1.512379e+06
              5                  1.250288e+06
Name: value, dtype: float64

In [16]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if has_simulations:
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,141,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,141,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,141,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,141,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,zero,0,129,0.0
539996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,zero,0,129,0.0
539997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,zero,0,129,0.0
539998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,zero,0,129,0.0


In [17]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders',
       'maternal_disorders_to_recovered_from_maternal_disorders'],
      dtype='str')

In [18]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.951651e+06
              2                  1.753006e+06
              3                  2.221155e+06
              4                  1.986394e+06
              5                  1.265697e+06
intervention  1                  2.918864e+06
              2                  1.733300e+06
              3                  2.196380e+06
              4                  1.965406e+06
              5                  1.253378e+06
zero          1                  3.016738e+06
              2                  1.788309e+06
              3                  2.261620e+06
              4                  2.020170e+06
              5                  1.276941e+06
Name: value, dtype: float64

In [19]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [20]:
path = f"{rescaled_child_root}/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if has_simulations:
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"{rescaled_child_root}/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,464.948091
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,366.176735
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,317.995586
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,245.723862
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,252.951034
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,40.953977
23996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,24.090575
23997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,40.953977
23998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,31.317747


In [21]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  199850.589385
              2                  165225.206405
              3                  146482.739308
              4                  139575.971549
              5                  132201.846641
intervention  1                  199539.820972
              2                  165001.164061
              3                  146306.878113
              4                  139366.383550
              5                  132071.757538
zero          1                  200245.674809
              2                  165485.384612
              3                  146680.282021
              4                  139819.286353
              5                  132286.163653
Name: value, dtype: float64

In [22]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [23]:
path = f"{anemia_root}/{vehicle}/{location}/anemia_cases.parquet"
if has_anemia_model:
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"{anemia_root}/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,39523.914887,zero
1,Female,0.0,0.019178,2,32531.761673,zero
2,Female,0.0,0.019178,3,28477.216975,zero
3,Female,0.0,0.019178,4,25072.633624,zero
4,Female,0.0,0.019178,5,18900.992888,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,10684.543485,intervention
746,Male,95.0,125.000000,2,10601.099169,intervention
747,Male,95.0,125.000000,3,10750.041826,intervention
748,Male,95.0,125.000000,4,10963.314118,intervention


In [24]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.257631e+08
              2                  1.182782e+08
              3                  1.166487e+08
              4                  1.111195e+08
              5                  1.052391e+08
intervention  1                  1.218458e+08
              2                  1.143149e+08
              3                  1.126337e+08
              4                  1.073045e+08
              5                  1.015031e+08
zero          1                  1.305852e+08
              2                  1.227763e+08
              3                  1.207382e+08
              4                  1.148301e+08
              5                  1.072487e+08
Name: value, dtype: float64

In [25]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.281941e+08
              2                  1.202010e+08
              3                  1.182793e+08
              4                  1.125546e+08
              5                  1.064500e+08
intervention  1                  1.242094e+08
              2                  1.161827e+08
              3                  1.142102e+08
              4                  1.086908e+08
              5                  1.026704e+08
zero          1                  1.331507e+08
              2                  1.247946e+08
              3                  1.224546e+08
              4                  1.163425e+08
              5                  1.084990e+08
Name: value, dtype: float64

In [26]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [27]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if has_ntd_model:
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  3792.235805
              2                  3350.044564
              3                  3007.552674
              4                  2835.973953
              5                  2415.279969
baseline      1                  3660.124523
              2                  3255.197855
              3                  2928.009576
              4                  2766.524686
              5                  2390.574135
intervention  1                  2341.801328
              2                  2194.109249
              3                  1978.660337
              4                  1900.736595
              5                  1869.849402
Name: value, dtype: float64

In [28]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)